# LeetCode 400 — Nth Digit bằng Python

Link đề: https://leetcode.com/problems/nth-digit/ · Độ khó: **Medium** · Chủ đề: Math, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def findNthDigit(self, n: int) -> int:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; mọi ví dụ và đáp án trong notebook được chạy lại và đối chiếu với một oracle "xây chuỗi thật" độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: nối các số `1, 2, 3, ...` thành một dãy chữ số vô hạn `123456789101112...`, hãy trả về chữ số thứ `n` (đếm từ `1`) của dãy đó.
2. Nắm công cụ Python: `str(x)` và chỉ số chuỗi, `len(str(x))`, lũy thừa `10 ** k`, `//` và `%`, `divmod`, vòng lặp `while`, `itertools.count`, `bisect`, và quan hệ giữa chỉ số đếm từ `0` với đếm từ `1`.
3. Đi từ cách **xây chuỗi thật `O(n)`** → **trừ dần từng khối số cùng độ dài `O(log n)`** → **tìm kiếm nhị phân trên số cần tìm** → **bảng mốc + `bisect`**.
4. Hiểu mấu chốt: các số có đúng `k` chữ số có `9 * 10^(k-1)` số, đóng góp `k * 9 * 10^(k-1)` chữ số; **trừ lần lượt các khối** để biết `n` rơi vào khối nào, sau đó dùng phép chia lấy nguyên/dư để biết **số nào** và **chữ số thứ mấy của số đó**.
5. Tránh các lỗi kinh điển: nhầm chỉ số `0`/`1` (quên `n - 1`), dùng `<=` thay `<` trong điều kiện dừng, quên nhân `cnt` lên 10 hoặc tăng `k`, tràn số `int` ở ngôn ngữ 32-bit khi tính `k * cnt`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Hãy trả về **chữ số thứ `n`** (đếm từ `1`) của dãy vô hạn các số nguyên `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, ...]` khi viết liền nhau thành một chuỗi chữ số.

**Ví dụ 1:** `n = 3` → `3`.

**Ví dụ 2:** `n = 11` → `0`. Dãy chữ số là `1 2 3 4 5 6 7 8 9 1 0 1 1 ...`; chữ số thứ `11` là `0`, nằm trong số `10`.

**Ràng buộc:** `1 <= n <= 2^31 - 1`.

In [1]:
from bisect import bisect_left
from itertools import count
import random
import time
from typing import List

examples = [
    (3, 3),
    (11, 0),
]
seq = "".join(str(i) for i in range(1, 16))
print("15 số đầu nối lại:", seq)
for n, expected in examples:
    print(f"chữ số thứ {n} của '{seq}' là {seq[n - 1]} | kỳ vọng {expected}")

15 số đầu nối lại: 123456789101112131415
chữ số thứ 3 của '123456789101112131415' là 3 | kỳ vọng 3
chữ số thứ 11 của '123456789101112131415' là 0 | kỳ vọng 0


### 2.1. Đọc kỹ đề

- Dãy bắt đầu từ `1` (không có `0` ở đầu). Chữ số được **đếm từ `1`**, nhưng chuỗi Python **đếm từ `0`**: chữ số thứ `n` là `s[n - 1]`.
- `n` có thể lên tới `2147483647`. Chuỗi tương ứng dài hơn **2 tỉ** ký tự — không thể dựng ra. Phải dùng công thức, cỡ `O(log n)`.
- Trả về một **chữ số** (số nguyên `0..9`), không phải ký tự.

## 3. Kiến thức nền

### 3.1. `str(x)`, chỉ số chuỗi và `len`

`str(370)` cho chuỗi `'370'`; `'370'[0]` là `'3'`; `int('3')` đổi ngược về số. `len(str(x))` là **số chữ số** của `x`. Chỉ số chuỗi bắt đầu từ `0`.

In [2]:
x = 370
s = str(x)
print(s, len(s), s[0], s[2], int(s[1]))
print([len(str(v)) for v in (7, 10, 99, 100, 12345)])

370 3 3 0 7
[1, 2, 2, 3, 5]


### 3.2. Lũy thừa của 10 và các khối số cùng độ dài

`10 ** k` là `1` theo sau `k` số `0`. Số nhỏ nhất có `k` chữ số là `10 ** (k - 1)`, số lớn nhất là `10 ** k - 1`. Do đó có `10 ** k - 10 ** (k - 1) = 9 * 10 ** (k - 1)` số có đúng `k` chữ số: `9` số một chữ số, `90` số hai chữ số, `900` số ba chữ số, ...

In [3]:
for k in range(1, 6):
    lo, hi = 10 ** (k - 1), 10 ** k - 1
    print(f"k={k}: từ {lo} đến {hi} -> {hi - lo + 1} số (công thức 9*10^(k-1) = {9 * 10 ** (k - 1)})")

k=1: từ 1 đến 9 -> 9 số (công thức 9*10^(k-1) = 9)
k=2: từ 10 đến 99 -> 90 số (công thức 9*10^(k-1) = 90)
k=3: từ 100 đến 999 -> 900 số (công thức 9*10^(k-1) = 900)
k=4: từ 1000 đến 9999 -> 9000 số (công thức 9*10^(k-1) = 9000)
k=5: từ 10000 đến 99999 -> 90000 số (công thức 9*10^(k-1) = 90000)


### 3.3. `//`, `%` và `divmod`

`a // b` là thương nguyên, `a % b` là phần dư, `divmod(a, b)` trả cả hai. Ở bài này: nếu mỗi số chiếm `k` chữ số thì vị trí `p` (đếm từ `0`) trong khối thuộc **số thứ `p // k`** (đếm từ `0`) và **chữ số thứ `p % k`** của số đó.

In [4]:
k = 3
for p in [0, 1, 2, 3, 4, 5, 6, 7]:
    q, r = divmod(p, k)
    print(f"vị trí p={p}: thuộc số thứ {q} của khối, chữ số thứ {r}")

vị trí p=0: thuộc số thứ 0 của khối, chữ số thứ 0
vị trí p=1: thuộc số thứ 0 của khối, chữ số thứ 1
vị trí p=2: thuộc số thứ 0 của khối, chữ số thứ 2
vị trí p=3: thuộc số thứ 1 của khối, chữ số thứ 0
vị trí p=4: thuộc số thứ 1 của khối, chữ số thứ 1
vị trí p=5: thuộc số thứ 1 của khối, chữ số thứ 2
vị trí p=6: thuộc số thứ 2 của khối, chữ số thứ 0
vị trí p=7: thuộc số thứ 2 của khối, chữ số thứ 1


### 3.4. Số chữ số của dãy các khối

Khối `k` chữ số đóng góp `k * 9 * 10 ** (k - 1)` chữ số. Bảng tích lũy cho biết mốc `n` mà mỗi khối kết thúc.

In [5]:
total = 0
print(" k |   số lượng số | chữ số của khối | tổng chữ số tới hết khối")
for k in range(1, 10):
    cnt = 9 * 10 ** (k - 1)
    digits = k * cnt
    total += digits
    print(f"{k:>2} | {cnt:>13,} | {digits:>15,} | {total:>24,}")
print("2^31 - 1 =", f"{2**31 - 1:,}", "-> nằm trong khối k = 9 (tổng tới hết khối 8 là 788,888,889)")

 k |   số lượng số | chữ số của khối | tổng chữ số tới hết khối
 1 |             9 |               9 |                        9
 2 |            90 |             180 |                      189
 3 |           900 |           2,700 |                    2,889
 4 |         9,000 |          36,000 |                   38,889
 5 |        90,000 |         450,000 |                  488,889
 6 |       900,000 |       5,400,000 |                5,888,889
 7 |     9,000,000 |      63,000,000 |               68,888,889
 8 |    90,000,000 |     720,000,000 |              788,888,889
 9 |   900,000,000 |   8,100,000,000 |            8,888,888,889
2^31 - 1 = 2,147,483,647 -> nằm trong khối k = 9 (tổng tới hết khối 8 là 788,888,889)


### 3.5. Tìm kiếm nhị phân và `bisect`

Nếu một hàm `f(x)` **không giảm** theo `x`, ta tìm `x` nhỏ nhất thỏa `f(x) >= n` bằng nhị phân trong `O(log)` bước. Thư viện `bisect_left(arr, v)` làm điều đó trên mảng đã sắp xếp: cho vị trí đầu tiên mà `arr[pos] >= v`.

In [6]:
marks = [9, 189, 2889, 38889]
print(bisect_left(marks, 5), bisect_left(marks, 9), bisect_left(marks, 10), bisect_left(marks, 190))   # 0 0 1 2

0 0 1 2


## 4. Ý tưởng nền tảng: tính tay

Tìm chữ số thứ `n = 15` của `123456789101112131415...`

1. `1..9` chiếm `9` chữ số đầu. Còn lại `15 - 9 = 6` chữ số, đếm bên trong khối các số **hai chữ số** `10, 11, 12, ...`.
2. Mỗi số chiếm `2` chữ số: chữ số thứ `6` (đếm từ `1`) → vị trí `5` (đếm từ `0`). `5 // 2 = 2` ⇒ số thứ `2` của khối (đếm từ `0`) là `10 + 2 = 12`; `5 % 2 = 1` ⇒ chữ số thứ `1` của `12` là `'2'`.
3. Kiểm tra bằng cách viết ra: `1 2 3 4 5 6 7 8 9 | 1 0 | 1 1 | 1 2 | ...` — chữ số thứ `15` là chữ số thứ hai của `12`, tức `2` ✓.

**Quy trình tổng quát:** (a) trừ dần các khối đầy đủ cho tới khi `n` nằm trong một khối; (b) trong khối đó, dùng `//` để biết **số nào**, `%` để biết **chữ số nào**.

## 5. Cách 1 — Brute Force: xây chuỗi thật

**Ý tưởng:** nối `"1" + "2" + "3" + ...` cho tới khi chuỗi đủ dài (`>= n` ký tự), rồi lấy `s[n - 1]`. Vì mỗi số góp ít nhất một chữ số, chỉ cần nối các số `1..n`.

Ta cũng có phiên bản **không tốn bộ nhớ**: duyệt số `1, 2, 3, ...`, trừ dần độ dài của từng số khỏi `n` (chỉ giữ biến đếm).

In [7]:
def nth_digit_string(n):
    s = "".join(str(i) for i in range(1, n + 1))       # chắc chắn đủ dài (>= n chữ số)
    return int(s[n - 1])


def nth_digit_stream(n):
    for num in count(1):                              # 1, 2, 3, ...
        for ch in str(num):
            n -= 1
            if n == 0:
                return int(ch)


for n, expected in examples:
    a, b = nth_digit_string(n), nth_digit_stream(n)
    print(n, "->", a, b, "| kỳ vọng", expected)
    assert a == b == expected

3 -> 3 3 | kỳ vọng 3
11 -> 0 0 | kỳ vọng 0


**Độ phức tạp:** `nth_digit_string`: Time `O(n)` và Space `O(n)`; `nth_digit_stream`: Time `O(n)` (duyệt tới `n` chữ số), Space `O(1)`.

**Nút thắt:** đo thời gian của phiên bản không tốn bộ nhớ cho vài `n` để thấy tốc độ tăng tuyến tính; với `n = 2^31 - 1` ta cần duyệt khoảng 2 tỉ chữ số — hàng chục phút trong Python.

In [8]:
for n in [10**4, 10**5, 10**6]:
    t = time.perf_counter()
    d = nth_digit_stream(n)
    dt = time.perf_counter() - t
    print(f"n = {n:>8,}: chữ số = {d}, xong dưới 2 giây? {dt < 2.0}")
est = (2**31 - 1) / 10**6
print(f"n = 2^31 - 1 chậm gấp khoảng {est:,.0f} lần n = 10^6 (tuyến tính) — không khả thi")

n =   10,000: chữ số = 7, xong dưới 2 giây? True
n =  100,000: chữ số = 2, xong dưới 2 giây? True
n = 1,000,000: chữ số = 1, xong dưới 2 giây? True
n = 2^31 - 1 chậm gấp khoảng 2,147 lần n = 10^6 (tuyến tính) — không khả thi


## 6. Ý tưởng thuật toán / bước đột phá

Dãy chữ số chia thành các **khối** theo số chữ số của các số: khối `k` gồm mọi số có đúng `k` chữ số, độ dài khối là `k * 9 * 10 ** (k - 1)` chữ số — **tính được ngay bằng công thức**, không cần liệt kê.

Vậy ta có thể **nhảy qua cả khối**:

- Nếu `n` lớn hơn độ dài khối `k` thì **trừ** độ dài đó khỏi `n` và xét khối `k + 1`.
- Khi `n <= độ dài khối k`, chữ số cần tìm nằm trong khối `k`. Lúc đó `n` là vị trí (đếm từ `1`) **bên trong khối**.

Trong khối `k`: vị trí đếm từ `0` là `p = n - 1`. Số chứa chữ số này là `num = 10 ** (k - 1) + p // k` (số đầu khối cộng số thứ tự trong khối) và chữ số cần lấy là `str(num)[p % k]`.

Thử bằng `n = 11`: `n > 9` nên trừ khối 1: `n = 2`, sang `k = 2`. `2 <= 180` nên dừng. `p = 1`, `num = 10 + 1 // 2 = 10`, `str(10)[1 % 2] = '0'` ✓. Hãy kiểm chứng công thức này trên nhiều `n` nhỏ bằng chuỗi thật.

In [9]:
def locate(n):
    """Trả về (k, num, idx): chữ số thứ n nằm ở số num (có k chữ số), tại vị trí idx của num (đếm từ 0)."""
    k, cnt = 1, 9
    while k * cnt < n:
        n -= k * cnt
        k += 1
        cnt *= 10
    num = 10 ** (k - 1) + (n - 1) // k
    idx = (n - 1) % k
    return k, num, idx


big = "".join(str(i) for i in range(1, 2000))
for n in [1, 9, 10, 11, 15, 189, 190, 191, 192, 193, 2889, 2890, 5000]:
    k, num, idx = locate(n)
    assert str(num)[idx] == big[n - 1]
    print(f"n={n:>5}: nằm trong số {num:>4} (k={k}) tại vị trí {idx} -> chữ số {str(num)[idx]} (chuỗi thật: {big[n - 1]})")

n=    1: nằm trong số    1 (k=1) tại vị trí 0 -> chữ số 1 (chuỗi thật: 1)
n=    9: nằm trong số    9 (k=1) tại vị trí 0 -> chữ số 9 (chuỗi thật: 9)
n=   10: nằm trong số   10 (k=2) tại vị trí 0 -> chữ số 1 (chuỗi thật: 1)
n=   11: nằm trong số   10 (k=2) tại vị trí 1 -> chữ số 0 (chuỗi thật: 0)
n=   15: nằm trong số   12 (k=2) tại vị trí 1 -> chữ số 2 (chuỗi thật: 2)
n=  189: nằm trong số   99 (k=2) tại vị trí 1 -> chữ số 9 (chuỗi thật: 9)
n=  190: nằm trong số  100 (k=3) tại vị trí 0 -> chữ số 1 (chuỗi thật: 1)
n=  191: nằm trong số  100 (k=3) tại vị trí 1 -> chữ số 0 (chuỗi thật: 0)
n=  192: nằm trong số  100 (k=3) tại vị trí 2 -> chữ số 0 (chuỗi thật: 0)
n=  193: nằm trong số  101 (k=3) tại vị trí 0 -> chữ số 1 (chuỗi thật: 1)
n= 2889: nằm trong số  999 (k=3) tại vị trí 2 -> chữ số 9 (chuỗi thật: 9)
n= 2890: nằm trong số 1000 (k=4) tại vị trí 0 -> chữ số 1 (chuỗi thật: 1)
n= 5000: nằm trong số 1527 (k=4) tại vị trí 2 -> chữ số 2 (chuỗi thật: 2)


## 7. Các cách tiếp theo

### Cách 2 — Trừ dần từng khối (toán học, chốt)

**Giải thích từng đoạn:**

1. `k = 1` (số chữ số của khối), `cnt = 9` (số lượng số trong khối), độ dài khối là `k * cnt`.
2. `while k * cnt < n`: `n` vượt khỏi khối hiện tại ⇒ `n -= k * cnt`, `k += 1`, `cnt *= 10`.
3. Thoát vòng lặp: `n` nằm trong khối `k` (vị trí đếm từ `1`). `num = 10 ** (k - 1) + (n - 1) // k`, `idx = (n - 1) % k`.
4. Trả về `int(str(num)[idx])`.

Điều kiện dừng dùng `<` (chứ không `<=`) vì khi `n` **đúng bằng** độ dài khối, chữ số cần tìm là chữ số **cuối cùng** của khối hiện tại (ví dụ `n = 9` là chữ số `9`).

In [10]:
def nth_digit_blocks(n):
    k, cnt = 1, 9
    while k * cnt < n:
        n -= k * cnt
        k += 1
        cnt *= 10
    num = 10 ** (k - 1) + (n - 1) // k
    idx = (n - 1) % k
    return int(str(num)[idx])


print([nth_digit_blocks(n) for n, _ in examples], nth_digit_blocks(2**31 - 1))

[3, 0] 2


In [11]:
# Code nộp LeetCode — Cách 2 (O(log n) time, O(1) space)
class SolutionBlocks:
    def findNthDigit(self, n: int) -> int:
        k, cnt = 1, 9
        while k * cnt < n:
            n -= k * cnt
            k += 1
            cnt *= 10
        num = 10 ** (k - 1) + (n - 1) // k
        idx = (n - 1) % k
        return int(str(num)[idx])


print([SolutionBlocks().findNthDigit(n) for n, _ in examples])

[3, 0]


**Độ phức tạp Cách 2:** vòng `while` chạy tối đa `9` lần (độ dài số `<= 10` chữ số) ⇒ Time `O(log n)`, Space `O(1)`.

### Cách 3 — Tìm kiếm nhị phân trên "số cần tìm"

**Giải thích từng đoạn:**

1. Định nghĩa `digits_upto(x)` = **tổng số chữ số** khi viết các số `1..x`. Hàm này **tăng** theo `x`. Công thức: với `L = len(str(x))`, tổng các khối đầy đủ `1..L-1` cộng `(x - 10 ** (L - 1) + 1) * L` cho khối cuối cùng chưa đầy.
2. Tìm số `x` **nhỏ nhất** sao cho `digits_upto(x) >= n` bằng nhị phân trên `[1, 10^9]` (đủ vì `n <= 2^31 - 1`).
3. Khi đó chữ số cần tìm nằm trong `x`: vị trí bên trong `x` là `n - digits_upto(x - 1) - 1` (đếm từ `0`).

In [12]:
def digits_upto(x):
    """Tổng số chữ số của dãy 1, 2, ..., x."""
    if x <= 0:
        return 0
    L = len(str(x))
    total = 0
    for k in range(1, L):
        total += k * 9 * 10 ** (k - 1)             # các khối đầy đủ
    total += (x - 10 ** (L - 1) + 1) * L           # khối cuối: các số L chữ số từ 10^(L-1) tới x
    return total


# kiểm chứng công thức với chuỗi thật
for x in [1, 9, 10, 11, 99, 100, 101, 999, 1000, 1500]:
    assert digits_upto(x) == len("".join(str(i) for i in range(1, x + 1))), x
print([digits_upto(x) for x in (9, 10, 99, 100, 999)])      # 9 11 189 192 2889

[9, 11, 189, 192, 2889]


In [13]:
def nth_digit_bsearch(n):
    lo, hi = 1, 10 ** 9
    while lo < hi:
        mid = (lo + hi) // 2
        if digits_upto(mid) >= n:                      # chữ số thứ n đã xuất hiện trong 1..mid
            hi = mid
        else:
            lo = mid + 1
    x = lo
    pos = n - digits_upto(x - 1) - 1                   # vị trí trong x, đếm từ 0
    return int(str(x)[pos])


print([nth_digit_bsearch(n) for n, _ in examples], nth_digit_bsearch(2**31 - 1))

[3, 0] 2


In [14]:
# Code nộp LeetCode — Cách 3 (nhị phân trên số, O(log^2 n) hoặc O(log n * 10))
class SolutionBinary:
    def findNthDigit(self, n: int) -> int:
        def digits_upto(x: int) -> int:
            if x <= 0:
                return 0
            length = len(str(x))
            total = sum(k * 9 * 10 ** (k - 1) for k in range(1, length))
            return total + (x - 10 ** (length - 1) + 1) * length

        lo, hi = 1, 10 ** 9
        while lo < hi:
            mid = (lo + hi) // 2
            if digits_upto(mid) >= n:
                hi = mid
            else:
                lo = mid + 1
        return int(str(lo)[n - digits_upto(lo - 1) - 1])


print([SolutionBinary().findNthDigit(n) for n, _ in examples])

[3, 0]


**Độ phức tạp Cách 3:** nhị phân chạy khoảng `log2(10^9) ≈ 30` vòng; mỗi lần gọi `digits_upto` tốn `O(số chữ số) = O(10)`. Tổng `O(log V * log V)` với `V = 10^9`: rất nhanh. Space `O(1)`.

### Cách 4 — Bảng mốc + `bisect`

**Giải thích từng đoạn:**

1. Tính trước mảng `ends[k - 1]` = tổng số chữ số tới hết khối `k` (`9, 189, 2889, ...`).
2. Với `n`, dùng `bisect_left(ends, n)` để tìm **chỉ số khối** (khối đầu tiên có `ends >= n`), suy ra `k = pos + 1`.
3. Phần còn lại trong khối: `rem = n - ends[pos - 1]` (với `pos = 0` thì `ends[-1]` bị coi là `0`). Sau đó dùng lại công thức `num`/`idx` của Cách 2.

In [15]:
ENDS = []
_total = 0
for _k in range(1, 11):
    _total += _k * 9 * 10 ** (_k - 1)
    ENDS.append(_total)
print(ENDS[:5], "...", ENDS[8], ENDS[9])


def nth_digit_bisect(n):
    pos = bisect_left(ENDS, n)                  # chỉ số khối
    k = pos + 1
    before = ENDS[pos - 1] if pos > 0 else 0    # số chữ số của các khối đứng trước
    rem = n - before                            # vị trí trong khối, đếm từ 1
    num = 10 ** (k - 1) + (rem - 1) // k
    return int(str(num)[(rem - 1) % k])


print([nth_digit_bisect(n) for n, _ in examples], nth_digit_bisect(2**31 - 1))

[9, 189, 2889, 38889, 488889] ... 8888888889 98888888889
[3, 0] 2


In [16]:
# Code nộp LeetCode — Cách 4 (bảng mốc + bisect)
from bisect import bisect_left


class SolutionBisect:
    def findNthDigit(self, n: int) -> int:
        ends, total = [], 0
        for k in range(1, 11):
            total += k * 9 * 10 ** (k - 1)
            ends.append(total)
        pos = bisect_left(ends, n)
        k = pos + 1
        rem = n - (ends[pos - 1] if pos > 0 else 0)
        num = 10 ** (k - 1) + (rem - 1) // k
        return int(str(num)[(rem - 1) % k])


print([SolutionBisect().findNthDigit(n) for n, _ in examples])

[3, 0]


**Độ phức tạp Cách 4:** Time `O(log log n)` cho `bisect` (mảng chỉ 10 phần tử) cộng `O(1)`; Space `O(1)` (mảng 10 phần tử). Thực chất là Cách 2 viết lại với bảng tính sẵn; tiện khi gọi hàm **rất nhiều lần**.

## 8. TRACE từng bước

Trace Cách 2 với `n = 1000`:

| vòng | `k` | `cnt` | `k * cnt` | `n` trước | điều kiện `k*cnt < n` | `n` sau |
|---|---|---|---|---|---|---|
| 1 | 1 | 9 | 9 | 1000 | đúng | `1000 - 9 = 991` |
| 2 | 2 | 90 | 180 | 991 | đúng | `991 - 180 = 811` |
| 3 | 3 | 900 | 2700 | 811 | sai ⇒ dừng | 811 |

Khi dừng: `k = 3`, `n = 811`; `num = 10^2 + (811 - 1) // 3 = 100 + 270 = 370`; `idx = (811 - 1) % 3 = 0` ⇒ `str(370)[0] = '3'`.

Hàm `trace_blocks` in đúng bảng trên. `trace_bsearch` in từng bước của nhị phân (Cách 3).

In [17]:
def trace_blocks(n):
    print("n gốc =", n)
    k, cnt = 1, 9
    print(f"{'k':>2} | {'cnt':>10} | {'k*cnt':>11} | {'n':>12} | quyết định")
    while True:
        block = k * cnt
        if block < n:
            print(f"{k:>2} | {cnt:>10} | {block:>11} | {n:>12} | n > khối -> trừ {block}")
            n -= block
            k += 1
            cnt *= 10
        else:
            print(f"{k:>2} | {cnt:>10} | {block:>11} | {n:>12} | n <= khối -> dừng")
            break
    num = 10 ** (k - 1) + (n - 1) // k
    idx = (n - 1) % k
    print(f"num = 10^{k - 1} + ({n} - 1) // {k} = {num};  idx = ({n} - 1) % {k} = {idx};  str(num) = '{num}' -> '{str(num)[idx]}'")
    return int(str(num)[idx])


print("kết quả:", trace_blocks(1000))

n gốc = 1000
 k |        cnt |       k*cnt |            n | quyết định
 1 |          9 |           9 |         1000 | n > khối -> trừ 9
 2 |         90 |         180 |          991 | n > khối -> trừ 180
 3 |        900 |        2700 |          811 | n <= khối -> dừng
num = 10^2 + (811 - 1) // 3 = 370;  idx = (811 - 1) % 3 = 0;  str(num) = '370' -> '3'
kết quả: 3


In [18]:
print("kết quả:", trace_blocks(11))
print()
print("kết quả:", trace_blocks(9))             # n đúng bằng độ dài khối 1: chữ số cuối của khối
print()
print("kết quả:", trace_blocks(2**31 - 1))

n gốc = 11
 k |        cnt |       k*cnt |            n | quyết định
 1 |          9 |           9 |           11 | n > khối -> trừ 9
 2 |         90 |         180 |            2 | n <= khối -> dừng
num = 10^1 + (2 - 1) // 2 = 10;  idx = (2 - 1) % 2 = 1;  str(num) = '10' -> '0'
kết quả: 0

n gốc = 9
 k |        cnt |       k*cnt |            n | quyết định
 1 |          9 |           9 |            9 | n <= khối -> dừng
num = 10^0 + (9 - 1) // 1 = 9;  idx = (9 - 1) % 1 = 0;  str(num) = '9' -> '9'
kết quả: 9

n gốc = 2147483647
 k |        cnt |       k*cnt |            n | quyết định
 1 |          9 |           9 |   2147483647 | n > khối -> trừ 9
 2 |         90 |         180 |   2147483638 | n > khối -> trừ 180
 3 |        900 |        2700 |   2147483458 | n > khối -> trừ 2700
 4 |       9000 |       36000 |   2147480758 | n > khối -> trừ 36000
 5 |      90000 |      450000 |   2147444758 | n > khối -> trừ 450000
 6 |     900000 |     5400000 |   2146994758 | n > khối -> trừ 5400000

In [19]:
def trace_bsearch(n):
    print("n =", n)
    lo, hi = 1, 10 ** 9
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        d = digits_upto(mid)
        step += 1
        keep = "hi = mid" if d >= n else "lo = mid + 1"
        if step <= 8 or hi - lo < 4:
            print(f"bước {step:>2}: lo={lo:>10}, hi={hi:>10}, mid={mid:>10}, digits_upto(mid)={d:>11} -> {keep}")
        if d >= n:
            hi = mid
        else:
            lo = mid + 1
    x = lo
    pos = n - digits_upto(x - 1) - 1
    print(f"... dừng sau {step} bước: x = {x}, vị trí trong x = {pos}, chữ số = {str(x)[pos]}")
    return int(str(x)[pos])


print("kết quả:", trace_bsearch(1000))

n = 1000
bước  1: lo=         1, hi=1000000000, mid= 500000000, digits_upto(mid)= 4388888898 -> hi = mid
bước  2: lo=         1, hi= 500000000, mid= 250000000, digits_upto(mid)= 2138888898 -> hi = mid
bước  3: lo=         1, hi= 250000000, mid= 125000000, digits_upto(mid)= 1013888898 -> hi = mid
bước  4: lo=         1, hi= 125000000, mid=  62500000, digits_upto(mid)=  488888897 -> hi = mid
bước  5: lo=         1, hi=  62500000, mid=  31250000, digits_upto(mid)=  238888897 -> hi = mid
bước  6: lo=         1, hi=  31250000, mid=  15625000, digits_upto(mid)=  113888897 -> hi = mid
bước  7: lo=         1, hi=  15625000, mid=   7812500, digits_upto(mid)=   53576396 -> hi = mid
bước  8: lo=         1, hi=   7812500, mid=   3906250, digits_upto(mid)=   26232646 -> hi = mid
bước 29: lo=       367, hi=       370, mid=       368, digits_upto(mid)=        996 -> lo = mid + 1
bước 30: lo=       369, hi=       370, mid=       369, digits_upto(mid)=        999 -> lo = mid + 1
... dừng sau 30 bước: x

## 9. Code nộp LeetCode

Lời giải chính của notebook là **Cách 2** (trừ dần các khối): ngắn, nhanh `O(log n)`, `O(1)` bộ nhớ.

In [20]:
class Solution:
    def findNthDigit(self, n: int) -> int:
        k, cnt = 1, 9                          # k: số chữ số của khối; cnt: số lượng số trong khối
        while k * cnt < n:                     # n vượt khỏi khối hiện tại -> bỏ qua cả khối
            n -= k * cnt
            k += 1
            cnt *= 10
        num = 10 ** (k - 1) + (n - 1) // k     # số chứa chữ số cần tìm
        idx = (n - 1) % k                      # vị trí trong số đó (đếm từ 0)
        return int(str(num)[idx])


for n, expected in examples:
    got = Solution().findNthDigit(n)
    print(n, "->", got)
    assert got == expected

3 -> 3
11 -> 0


## 10. Vì sao đúng

1. **Khối:** các số có `k` chữ số liên tiếp nhau trong dãy từ `10^(k-1)` đến `10^k - 1`, không xen kẽ với số có độ dài khác. Mỗi khối đóng góp đúng `k * 9 * 10^(k-1)` chữ số liền nhau.
2. **Trừ khối:** mỗi lần `n > k * cnt`, `n`-th digit chắc chắn nằm sau khối này; trừ độ dài khối giữ nguyên ý nghĩa "vị trí đếm từ 1 tính từ đầu khối kế tiếp".
3. **Dừng đúng:** khi `n <= k * cnt`, chữ số cần tìm nằm trong khối `k`; `n >= 1` luôn đúng vì ta chỉ trừ khi `n > k * cnt`, nên `n - k*cnt >= 1`.
4. **Định vị:** vị trí đếm từ 0 trong khối là `p = n - 1`; mỗi số chiếm `k` ô nên số thứ `p // k` (từ 0) có giá trị `10^(k-1) + p // k` và chữ số thứ `p % k` của nó là đáp án.
5. **Số lần lặp:** `n <= 2^31 - 1 < 788,888,889 + 9 * 10^8 * 9` nên `k <= 9`.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Xây chuỗi / duyệt từng chữ số | `O(n)` | `O(n)` / `O(1)` |
| 2. Trừ dần khối | **`O(log n)`** | **`O(1)`** |
| 3. Nhị phân trên số | `O(log^2 n)` | `O(1)` |
| 4. Bảng mốc + `bisect` | `O(log log n)` | `O(1)` |

`log n` ở đây chính là **số chữ số** của số cần tìm (tối đa `9` hoặc `10` với `n <= 2^31 - 1`), nên thực tế đều là hằng số nhỏ.

## 12. Bộ test

Oracle độc lập: **chuỗi thật** `"123456789101112..."` được dựng bằng `"".join(map(str, range(1, M)))`, dài khoảng `5.9` triệu ký tự (đủ cho `n` tới `5,888,889`). Với `n` lớn hơn (tới `2^31 - 1`) ta đối chiếu giữa các cách + kiểm tra **bất biến**: số `x` tìm được phải thỏa `digits_upto(x - 1) < n <= digits_upto(x)`, trong đó `digits_upto` đã được kiểm chứng bằng chuỗi thật.

In [21]:
SOLUTIONS = [Solution(), SolutionBlocks(), SolutionBinary(), SolutionBisect()]

# 12.1 Ví dụ của đề + biên theo ranh giới các khối
boundaries = {
    1: 1, 9: 9, 10: 1, 11: 0, 12: 1, 13: 1, 14: 1, 15: 2,
    189: 9, 190: 1, 191: 0, 192: 0, 193: 1,          # hết khối 2 (99), bắt đầu 100
    2889: 9, 2890: 1, 2891: 0, 2892: 0, 2893: 0,     # hết khối 3 (999), bắt đầu 1000
}
for n, expected in boundaries.items():
    for sol in SOLUTIONS:
        got = sol.findNthDigit(n)
        assert got == expected, (type(sol).__name__, n, got, expected)
print("OK: ví dụ đề + biên các khối cho 4 lớp")

OK: ví dụ đề + biên các khối cho 4 lớp


In [22]:
# 12.2 Oracle chuỗi thật: đối chiếu MỌI n từ 1 đến 20000 cho 4 lớp, bằng nối chuỗi
M = 1_000_000
BIG = "".join(map(str, range(1, M)))
print("độ dài chuỗi thật:", len(BIG))
for n in range(1, 20001):
    expected = int(BIG[n - 1])
    for sol in SOLUTIONS:
        assert sol.findNthDigit(n) == expected, (type(sol).__name__, n)
print("OK: mọi n trong [1, 20000]")

độ dài chuỗi thật: 5888889


OK: mọi n trong [1, 20000]


In [23]:
# 12.3 Ngẫu nhiên có seed: n tới hết chuỗi thật (5.9 triệu), 4 lớp + hàm duyệt từng chữ số
random.seed(400)
for _ in range(20000):
    n = random.randint(1, len(BIG))
    expected = int(BIG[n - 1])
    assert Solution().findNthDigit(n) == expected, n
    assert SolutionBinary().findNthDigit(n) == expected, n
    assert SolutionBisect().findNthDigit(n) == expected, n
print("OK: 20000 n ngẫu nhiên trong [1, 5.9 triệu]")

OK: 20000 n ngẫu nhiên trong [1, 5.9 triệu]


In [24]:
# 12.4 n lớn đúng ràng buộc (tới 2^31 - 1): đối chiếu giữa các cách + bất biến digits_upto
for x in list(range(1, 1500)) + [10**5, 10**5 + 7, 999_999]:
    assert digits_upto(x) == len("".join(map(str, range(1, x + 1)))), x       # nền tảng của oracle

random.seed(401)
specials = [2**31 - 1, 2**31 - 2, 10**9, 788_888_889, 788_888_890, 788_888_888, 10**8, 10**8 + 1, 892_999_999]
samples = specials + [random.randint(1, 2**31 - 1) for _ in range(20000)]
for n in samples:
    results = {sol.findNthDigit(n) for sol in SOLUTIONS}
    assert len(results) == 1, (n, results)
    # bất biến: số x chứa chữ số thứ n
    k, num, idx = locate(n)
    assert digits_upto(num - 1) < n <= digits_upto(num), n
    assert digits_upto(num - 1) + idx + 1 == n
    assert str(num)[idx] == str(results.pop())
print("OK: cả", len(samples), "giá trị n lớn: 4 lớp thống nhất và thỏa bất biến")
print("2^31 - 1 ->", Solution().findNthDigit(2**31 - 1), "| 10^9 ->", Solution().findNthDigit(10**9))

OK: cả 20009 giá trị n lớn: 4 lớp thống nhất và thỏa bất biến
2^31 - 1 -> 2 | 10^9 -> 1


In [25]:
# 12.5 Phản ví dụ cho lỗi "dùng <= trong điều kiện dừng" (n đúng bằng độ dài khối) — xem mục 13
def wrong_le(n):
    k, cnt = 1, 9
    while k * cnt <= n:                        # SAI: nên là <
        n -= k * cnt
        k += 1
        cnt *= 10
    num = 10 ** (k - 1) + (n - 1) // k
    return int(str(num)[(n - 1) % k])


def safe(f, n):
    """Chạy f(n); nếu f ném IndexError thì trả 'lỗi' (để so sánh với đáp án thật mà không dừng chương trình)."""
    try:
        return f(n)
    except IndexError:
        return "IndexError"


bad = [n for n in range(1, 400) if safe(wrong_le, n) != int(BIG[n - 1])]
print("n sai với '<=':", bad[:8], "... tổng", len(bad), "giá trị trong 1..399")
print("n = 9 với '<=':", safe(wrong_le, 9), "| n = 189:", safe(wrong_le, 189))
assert bad and bad[0] == 9

n sai với '<=': [9, 189] ... tổng 2 giá trị trong 1..399
n = 9 với '<=': IndexError | n = 189: IndexError


In [26]:
# 12.6 Hiệu năng: 10^5 lần gọi với n lớn
random.seed(402)
ns = [random.randint(1, 2**31 - 1) for _ in range(10**5)]
t = time.perf_counter()
total = sum(Solution().findNthDigit(n) for n in ns)
dt = time.perf_counter() - t
assert dt < 5.0
print("10^5 lần gọi xong dưới 5 giây:", dt < 5.0, "| tổng các chữ số =", total)

10^5 lần gọi xong dưới 5 giây: True | tổng các chữ số = 422788


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm chỉ số 0/1 (quên trừ 1)

In [27]:
def wrong_off_by_one(n):
    k, cnt = 1, 9
    while k * cnt < n:
        n -= k * cnt
        k += 1
        cnt *= 10
    num = 10 ** (k - 1) + n // k               # SAI: phải là (n - 1) // k
    return int(str(num)[n % k])                # SAI: phải là (n - 1) % k


bad = [n for n in range(1, 100) if wrong_off_by_one(n) != int(BIG[n - 1])]
print("n sai đầu tiên:", bad[:6], "| ví dụ n = 11: sai", wrong_off_by_one(11), "đúng", Solution().findNthDigit(11))
assert bad and wrong_off_by_one(11) != Solution().findNthDigit(11)

n sai đầu tiên: [1, 2, 3, 4, 5, 6] | ví dụ n = 11: sai 1 đúng 0


**Vì sao sai:** `n` là vị trí **đếm từ 1**, còn `//`/`%` cần vị trí **đếm từ 0**. Ví dụ `n = 2` trong khối 2 chữ số: `2 // 2 = 1` chỉ số `11` thay vì `10`. **Cách sửa:** luôn đổi sang đếm từ 0 bằng `n - 1` trước khi chia.

### Lỗi 2: điều kiện dừng dùng `<=` thay vì `<`

Xem mục 12.5: khi `n` **bằng đúng** độ dài khối (ví dụ `n = 9`, `n = 189`), `<=` trừ nhầm cả khối khiến `n` thành `0`. Chữ số thứ `9` là `9` (cuối khối 1), nhưng code sai đi sang khối 2 với `n = 0`.

In [28]:
print("n = 9: sai =", safe(wrong_le, 9), "| đúng =", Solution().findNthDigit(9))
print("n = 189: sai =", safe(wrong_le, 189), "| đúng =", Solution().findNthDigit(189))
assert safe(wrong_le, 9) != 9 and safe(wrong_le, 189) != 9
assert Solution().findNthDigit(9) == 9 and Solution().findNthDigit(189) == 9

n = 9: sai = IndexError | đúng = 9
n = 189: sai = IndexError | đúng = 9


**Vì sao sai:** với `<=`, khi `n = 9`, `1 * 9 <= 9` đúng ⇒ `n = 0`, `k = 2`; `num = 10 + (0 - 1) // 2 = 10 + (-1) = 9` (do `//` làm tròn xuống số âm!), `(0 - 1) % 2 = 1` ⇒ `str(9)[1]` vượt chỉ số: **`IndexError`**. **Cách sửa:** dùng `<`; khi `n == k * cnt` ta vẫn ở trong khối hiện tại (chữ số cuối cùng).

### Lỗi 3: quên cập nhật `cnt *= 10` (hoặc `k += 1`)

In [29]:
def wrong_no_cnt_update(n):
    k, cnt = 1, 9
    while k * cnt < n:
        n -= k * cnt
        k += 1                                  # SAI: quên cnt *= 10 (cnt phải thành 90, 900, ...)
    num = 10 ** (k - 1) + (n - 1) // k
    return int(str(num)[(n - 1) % k])


for n in [11, 100, 1000]:
    print(f"n={n}: sai {wrong_no_cnt_update(n)} | đúng {Solution().findNthDigit(n)} | chuỗi thật {BIG[n - 1]}")
assert any(wrong_no_cnt_update(n) != int(BIG[n - 1]) for n in range(1, 1000))

n=11: sai 0 | đúng 0 | chuỗi thật 0
n=100: sai 1 | đúng 5 | chuỗi thật 5
n=1000: sai 0 | đúng 3 | chuỗi thật 3


**Vì sao sai:** nếu `cnt` vẫn là `9`, khối 2 chữ số bị coi là chỉ có `9` số (`2*9 = 18` chữ số) thay vì `90` số (`180` chữ số), nên ta nhảy sang khối `3` quá sớm. **Cách sửa:** cùng một lúc cập nhật cả `n`, `k` và `cnt`; khi viết lại hãy đọc to ba dòng đó.

### Lỗi 4: dựng cả chuỗi dài (hoặc `range(1, n + 1)`) cho `n` lớn

`"".join(str(i) for i in range(1, n + 1))` với `n = 2^31 - 1` sẽ tạo chuỗi **hơn 2 tỉ ký tự** (cỡ 2 GB) và vòng lặp 2 tỉ lần — hết bộ nhớ/thời gian. Ta chỉ đo ước lượng, không chạy:

In [30]:
n = 2**31 - 1
print(f"cần ít nhất {n:,} ký tự (khoảng {n / 1024**3:.1f} GiB, mỗi ký tự 1 byte) và {n:,} lần nối số")

cần ít nhất 2,147,483,647 ký tự (khoảng 2.0 GiB, mỗi ký tự 1 byte) và 2,147,483,647 lần nối số


### Lỗi 5: tràn số nguyên 32-bit khi tính `k * cnt`

Ở Java/C++ kiểu `int`, `k * cnt` với `k = 9`, `cnt = 900000000` bằng `8.1 * 10^9` **vượt** `2^31 - 1 ≈ 2.147 * 10^9`. Python không tràn, nhưng khi chuyển ngôn ngữ phải dùng `long`. Mô phỏng tràn:

In [31]:
def to_int32(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= (1 << 31) else x


k, cnt = 9, 900_000_000
print("k * cnt (Python) =", f"{k * cnt:,}", "| nếu là int32 =", to_int32(k * cnt))
assert to_int32(k * cnt) != k * cnt

k * cnt (Python) = 8,100,000,000 | nếu là int32 = -489934592


**Vì sao sai:** giá trị trung gian tràn thành số âm/rác khiến điều kiện `k * cnt < n` sai. **Cách sửa:** dùng `long`/`int64` (hoặc ép kiểu `(long) k * cnt`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "dãy vô hạn được xây theo quy luật, hỏi phần tử thứ `n` rất lớn" ⇒ **đếm theo khối** rồi **định vị bằng `//` và `%`**. Cùng họ: *Kth Symbol in Grammar* (779), *Find K-th Bit in Nth Binary String* (1545), *K-th Smallest in Lexicographical Order* (440).
- **Quy trình 3 bước:** (1) xác định khối bằng cách trừ; (2) tính số chứa vị trí đó; (3) lấy đúng chữ số.
- **Luôn đổi `n` về đếm từ 0** trước khi dùng `//` và `%`.
- **Kiểm chứng:** dựng chuỗi thật cho `n` nhỏ và so sánh với công thức (như mục 6 và 12).
- **Khi `n` nằm ngay mốc khối** (9, 189, 2889, ...) luôn thử riêng — đó là nơi lỗi `<` / `<=` lộ ra.
- **Đối chiếu bằng hàm đếm đơn điệu:** nếu có `digits_upto(x)`, nhị phân luôn dùng được (và dễ chứng minh đúng).

## 15. Biến thể

1. **Chiều ngược:** cho số `x`, tìm vị trí (đếm từ 1) của **chữ số đầu tiên** của `x` trong dãy: `digits_upto(x - 1) + 1`.
2. **Dãy bắt đầu từ `0`:** `0, 1, 2, ...` — dịch `n` đi một đơn vị (chữ số thứ nhất là `0`).
3. **Chỉ các số chẵn / lũy thừa của 2 / số nguyên tố:** mỗi khối cần đếm bằng công thức khác nhau.
4. **Cơ số khác (nhị phân, thập lục phân):** thay `10` bằng cơ số `b`, khối `k` có `(b - 1) * b^(k-1)` số.
5. **Tổng của `n` chữ số đầu tiên** hoặc **đếm chữ số `1` trong `n` chữ số đầu**: cần cộng thêm công thức đếm theo khối.

In [32]:
def position_of_number(x):
    """Vị trí (đếm từ 1) của chữ số đầu tiên của x trong dãy 123456789101112..."""
    return digits_upto(x - 1) + 1


for x in [1, 9, 10, 11, 99, 100, 370, 12345]:
    p = position_of_number(x)
    assert Solution().findNthDigit(p) == int(str(x)[0])
    print(f"số {x}: chữ số đầu ở vị trí {p}, chữ số thứ {p} = {Solution().findNthDigit(p)}")


def nth_digit_base(n, base=2):
    """Chữ số thứ n của dãy các số 1, 2, 3, ... viết trong cơ số base (<= 10 cho đơn giản)."""
    k, cnt = 1, base - 1
    while k * cnt < n:
        n -= k * cnt
        k += 1
        cnt *= base
    num = base ** (k - 1) + (n - 1) // k
    digits = []
    while num:
        digits.append(num % base)
        num //= base
    digits.reverse()
    return digits[(n - 1) % k]


seq2 = "".join(bin(i)[2:] for i in range(1, 200))       # 1 10 11 100 101 ...
assert all(nth_digit_base(n, 2) == int(seq2[n - 1]) for n in range(1, 500))
print("dãy nhị phân:", seq2[:20], "... OK")

số 1: chữ số đầu ở vị trí 1, chữ số thứ 1 = 1
số 9: chữ số đầu ở vị trí 9, chữ số thứ 9 = 9
số 10: chữ số đầu ở vị trí 10, chữ số thứ 10 = 1
số 11: chữ số đầu ở vị trí 12, chữ số thứ 12 = 1
số 99: chữ số đầu ở vị trí 188, chữ số thứ 188 = 9
số 100: chữ số đầu ở vị trí 190, chữ số thứ 190 = 1
số 370: chữ số đầu ở vị trí 1000, chữ số thứ 1000 = 3
số 12345: chữ số đầu ở vị trí 50615, chữ số thứ 50615 = 1
dãy nhị phân: 11011100101110111100 ... OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Xây chuỗi / duyệt chữ số | nối thật, hoặc trừ dần từng chữ số | `O(n)` | `O(n)` / `O(1)` | hiểu đề, `n` nhỏ, **oracle** |
| 2. Trừ dần khối | nhảy qua cả khối `k*9*10^(k-1)` | `O(log n)` | `O(1)` | **nộp bài**, phỏng vấn |
| 3. Nhị phân trên số | tìm `x` nhỏ nhất có `digits_upto(x) >= n` | `O(log^2 n)` | `O(1)` | khi có hàm đếm đơn điệu, dễ chứng minh |
| 4. Bảng mốc + `bisect` | tính sẵn mốc cuối mỗi khối | `O(log log n)` | `O(1)` | gọi hàm rất nhiều lần |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `len(str(100))`; (b) `str(370)[0]`; (c) `divmod(7, 3)`; (d) số lượng chữ số mà khối các số `4` chữ số đóng góp (`k * 9 * 10^(k-1)` với `k = 4`).

**Bài 2 (tính tay).** Tìm chữ số thứ `200` của dãy: trừ khối `1` và khối `2`, rồi xác định số và chữ số. Kiểm tra bằng cách liệt kê `100 101 102 103` vào chuỗi.

**Bài 3 (điền TODO).** Hoàn thành `nth_digit_todo(n)` theo Cách 2.

In [33]:
def nth_digit_todo(n):
    # TODO 1: khởi tạo k = 1 (số chữ số), cnt = 9 (số lượng số trong khối)
    # TODO 2: while k * cnt < n: trừ n -= k * cnt; tăng k; nhân cnt lên 10
    # TODO 3: num = 10 ** (k - 1) + (n - 1) // k
    # TODO 4: idx = (n - 1) % k
    # TODO 5: trả về int(str(num)[idx])
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `nth_digit_bs(n)` theo Cách 3 (nhị phân trên số, cần hàm `digits_upto`) **không nhìn lại** mục 7; kiểm tra với `[3, 11, 1000, 2**31 - 1]`.

**Bài 5 (biến thể / mini challenge).** Viết `position_of_number2(x)` trả về vị trí (đếm từ 1) của **chữ số đầu tiên** của `x` trong dãy, **không** dùng `digits_upto` mà dùng cách trừ khối (đi các `k` nhỏ hơn số chữ số của `x`). Kiểm tra: với mọi `x` trong `1..3000`, `Solution().findNthDigit(position_of_number2(x)) == int(str(x)[0])` và `position_of_number2(100) == 190`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3`; `'3'`; `(2, 1)`; `4 * 9000 = 36000`. Bài 3: xem hàm `nth_digit_blocks` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `200 - 9 = 191`, `191 - 180 = 11`; khối 3 chữ số, `num = 100 + (11 - 1) // 3 = 103`, `idx = (11 - 1) % 3 = 1` ⇒ `'0'`. Bài 4: nhị phân `lo, hi = 1, 10**9`; điều kiện `digits_upto(mid) >= n`.
- **Mức 3 (sâu):** Bài 5: tổng chữ số của các khối đầy đủ `k = 1 .. len(str(x)) - 1` rồi cộng `(x - 10 ** (L - 1)) * L + 1`. Đó chính là `digits_upto(x - 1) + 1`, nhưng bạn tự viết vòng `for` thay vì gọi hàm có sẵn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert len(str(100)) == 3
assert str(370)[0] == "3"
assert divmod(7, 3) == (2, 1)
assert 4 * 9 * 10 ** 3 == 36000

# Bài 2
n2 = 200
n2 -= 1 * 9
n2 -= 2 * 90
assert n2 == 11
num2 = 100 + (n2 - 1) // 3
idx2 = (n2 - 1) % 3
assert (num2, idx2) == (103, 1)
assert str(num2)[idx2] == "0" == str(BIG[199]) and Solution().findNthDigit(200) == 0
assert BIG[189:201] == "100101102103"[:12]


# Bài 3
def nth_digit_todo_ans(n):
    k, cnt = 1, 9
    while k * cnt < n:
        n -= k * cnt
        k += 1
        cnt *= 10
    num = 10 ** (k - 1) + (n - 1) // k
    idx = (n - 1) % k
    return int(str(num)[idx])


for n, expected in examples:
    assert nth_digit_todo_ans(n) == expected
assert all(nth_digit_todo_ans(n) == int(BIG[n - 1]) for n in range(1, 5000))


# Bài 4
def nth_digit_bs(n):
    lo, hi = 1, 10 ** 9
    while lo < hi:
        mid = (lo + hi) // 2
        if digits_upto(mid) >= n:
            hi = mid
        else:
            lo = mid + 1
    return int(str(lo)[n - digits_upto(lo - 1) - 1])


for n in [3, 11, 1000, 2**31 - 1]:
    assert nth_digit_bs(n) == Solution().findNthDigit(n)


# Bài 5
def position_of_number2(x):
    L = len(str(x))
    total = 0
    for k in range(1, L):
        total += k * 9 * 10 ** (k - 1)              # các khối đầy đủ trước khối của x
    return total + (x - 10 ** (L - 1)) * L + 1


assert position_of_number2(100) == 190
for x in range(1, 3001):
    assert Solution().findNthDigit(position_of_number2(x)) == int(str(x)[0])
    assert position_of_number2(x) == position_of_number(x)
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng

## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (trừ dần các khối số cùng độ dài)

```python
class Solution:
    def findNthDigit(self, n: int) -> int:
        k, cnt = 1, 9
        while k * cnt < n:
            n -= k * cnt
            k += 1
            cnt *= 10
        num = 10 ** (k - 1) + (n - 1) // k
        idx = (n - 1) % k
        return int(str(num)[idx])
```

**Checklist**

- [ ] Hiểu khối `k` có `9 * 10^(k-1)` số và `k * 9 * 10^(k-1)` chữ số?
- [ ] Vòng lặp dùng `k * cnt < n` (không phải `<=`)?
- [ ] Mỗi vòng cập nhật đủ **ba** thứ: `n -= k * cnt`, `k += 1`, `cnt *= 10`?
- [ ] Đổi sang đếm từ 0 bằng `n - 1` trước khi dùng `//` và `%`?
- [ ] Trả về `int(...)` (một chữ số), không phải ký tự?
- [ ] Với Java/C++: dùng `long` cho `k * cnt`; đã thử `n = 1, 9, 10, 11, 189, 190, 2^31 - 1`?